# Eishkom Stage 1: Cloud TPU v5e-1 Runner (Google Cloud & Colab TPU)
Open this notebook in Google Colab (Research or Enterprise) with hardware accelerator set to **TPU v5e-1** (1 chip = 16GB HBM, 197 TFLOPs bfloat16).

### Cloud TPU v5e-1 Architecture & Strategy:
- **Unsloth vs. TPU Note:** Unsloth uses custom NVIDIA Triton/CUDA kernels and does not run on TPUs. Therefore, this notebook runs native **PyTorch/XLA (`torch_xla`) + Hugging Face Transformers + PEFT LoRA**.
- **Hardware Optimization:** TPU v5e matrix units are purpose-built for **bfloat16**. Training runs natively in bfloat16 with `batch=8, grad_accum=2`, utilizing ~6–8 GB of the 16 GB High Bandwidth Memory (HBM).
- **Multi-Quant GGUF Export:** Converts adapter weights and exports **Q8_0, Q6_K, pure Q4_K, and Q4_K_M** using the x86 host CPU and llama.cpp.

In [ ]:
# 1. Verify Cloud TPU v5e Hardware
import os
print('PJRT Device:', os.environ.get('PJRT_DEVICE', 'Default'))
try:
    import torch_xla.core.xla_model as xm
    print('✅ Active TPU Device:', xm.xla_device())
except Exception as e:
    print('TPU status note:', e)


## 2. Setup Workspace & Unpack Kit
Upload `gariep_colab_research_kit.zip` (or `eishkom_colab_kit.zip`) to the **Files** panel on the left.

In [ ]:
# 2. Setup workspace & unpack kit
import os, shutil, zipfile

if os.path.exists('gariep_colab_research_kit.zip'):
    print('Unpacking gariep_colab_research_kit.zip...')
    with zipfile.ZipFile('gariep_colab_research_kit.zip', 'r') as z:
        z.extractall('.')
    print('✅ Gariep 2B Kit unpacked successfully!')
elif os.path.exists('eishkom_colab_kit.zip'):
    print('Unpacking eishkom_colab_kit.zip...')
    with zipfile.ZipFile('eishkom_colab_kit.zip', 'r') as z:
        z.extractall('.')
    print('✅ Eishkom Kit unpacked!')
else:
    print('TIP: Please drag and drop gariep_colab_research_kit.zip or eishkom_colab_kit.zip into Files!')

os.environ['SA_WORKSPACE'] = os.path.abspath('workspace')
print('Workspace active at:', os.environ['SA_WORKSPACE'])


## 3. Install TPU Training Tools (PyTorch-XLA + Transformers + PEFT)
Installs Hugging Face transformers, PEFT LoRA, accelerate, and mathematical verification packages.

In [ ]:
!pip install -q transformers peft accelerate datasets sympy pypdf matplotlib


## 4. Dry Run Smoke Test on TPU (5 seconds)
Verifies dataset tokens, chat templates, and loss masks before launching the full training run.

In [ ]:
!python training/train_sft.py --model empero-ai/Qwen3.8-2B-Distill --data workspace/build/r1 --no-unsloth --dry-run --show 2


## 5. TRAIN Gariep 2B on Cloud TPU v5e-1 (~15–20 minutes)
Trains `gariep-stage-1` (2B) with PyTorch/XLA bfloat16 LoRA across all 5,755 South African CAPS samples.
Hardware settings: `batch=8, grad_accum=2` saturates TPU v5e matrix units for ultra-fast throughput.

In [ ]:
# Gariep 2B: Full training run on TPU v5e
!python training/train_sft.py --model empero-ai/Qwen3.8-2B-Distill --data workspace/build/r1 --out workspace/runs/r1-gariep --no-unsloth --epochs 3 --batch 8 --grad-accum 2 --show 0


## 6. Compile llama.cpp on Host CPU for GGUF Quantization (~1 minute)
TPU VMs use standard Linux x86 CPUs. We compile llama.cpp natively to quantize the model into all GGUF formats.

In [ ]:
!git clone -q https://github.com/ggerganov/llama.cpp
!cd llama.cpp && cmake -B build -DGGML_CUDA=OFF > /dev/null && cmake --build build --config Release -j4 > /dev/null
print('✅ llama.cpp compiled successfully on host CPU!')


## 7. EXPORT GGUF Models (Multi-Quant: Q8_0, Q6_K, pure Q4_K, Q4_K_M)
Compiles all requested quantization formats directly from your trained adapter in one pass:
- **Q8_0**: Maximum 2B reasoning fidelity.
- **Q6_K**: High fidelity for budget PCs/laptops.
- **Q4_K**: Pure K-quantization variant without sub-modifiers.
- **Q4_K_M**: Mobile format for phones and low-RAM devices (~1.3 GB).

In [ ]:
!python training/export_gguf.py --adapter workspace/runs/r1-gariep/adapter --base empero-ai/Qwen3.8-2B-Distill --method manual --llama-cpp llama.cpp --out workspace/exports/gariep-stage-1 --quants q8_0 q6_k q4_k q4_k_m


## 8. Package & Download Models and Training Runs
Packages all GGUF models and adapter weights into clean archives for easy 1-click download.

In [ ]:
import os, zipfile
from google.colab import files

# 1. Package all exported GGUF models into one archive
gguf_zip = '/content/gariep_stage1_gguf_models.zip'
with zipfile.ZipFile(gguf_zip, 'w', zipfile.ZIP_STORED) as z:
    for root, dirs, files_list in os.walk('/content/workspace/exports'):
        for f in files_list:
            if f.endswith('.gguf'):
                p = os.path.join(root, f)
                z.write(p, arcname=os.path.basename(p))

# 2. Package adapter, loss plot, and training metrics
run_zip = '/content/gariep_stage1_adapter_and_loss.zip'
with zipfile.ZipFile(run_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, dirs, files_list in os.walk('/content/workspace/runs/r1-gariep'):
        for f in files_list:
            p = os.path.join(root, f)
            z.write(p, arcname=os.path.relpath(p, '/content/workspace'))

if os.path.exists(gguf_zip):
    print(f'✅ Created {gguf_zip} ({os.path.getsize(gguf_zip)/(1024*1024):.1f} MB)')
if os.path.exists(run_zip):
    print(f'✅ Created {run_zip} ({os.path.getsize(run_zip)/(1024*1024):.1f} MB)')
    print('⬇️ Downloading adapter & metrics archive to browser...')
    files.download(run_zip)

print('\nDone! To download the GGUF models, right-click gariep_stage1_gguf_models.zip in the Files panel or copy to Google Drive.')
